In [1]:

import numpy as np
import random

grid = []
for i in range(4):
    row = []
    for j in range(4):
        row.append(".")
    grid.append(row)

r = int(input("Enter the row for taxi start(0-3) : "))
c = int(input("Enter the column for taxi start(0-3) : "))
start = (r, c)
grid[r][c] = "S"

r = int(input("Enter the row for passenger(0-3) : "))
c = int(input("Enter the column for passenger(0-3) : "))
pickup = (r, c)
grid[r][c] = "P"

r = int(input("Enter the row for destination(0-3) : "))
c = int(input("Enter the column for destination(0-3) : "))
drop = (r, c)
grid[r][c] = "D"

print("\nTaxi Grid World :")
for row in grid:
    print(*row)

# Q-table: row, column, passenger status, action
Q = np.zeros((4, 4, 2, 6))

actions = [(-1,0), (1,0), (0,-1), (0,1)]

alpha = 0.1
gamma = 0.9
epsilon = 0.2

# Training
for episodes in range(5000):
    position = start
    passenger = 0

    for steps in range(100):
        r, c = position

        if random.random() < epsilon:
            action = random.randint(0, 5)
        else:
            action = np.argmax(Q[r, c, passenger])

        nr, nc = r, c
        new_passenger = passenger
        reward = -1
        done = False

        if action < 4:
            dr, dc = actions[action]
            nr, nc = r + dr, c + dc

            if not (0 <= nr < 4 and 0 <= nc < 4):
                nr, nc = r, c
                reward = -5

        elif action == 4:
            if position == pickup and passenger == 0:
                new_passenger = 1
                reward = 10
            else:
                reward = -10

        elif action == 5:
            if position == drop and passenger == 1:
                reward = 100
                done = True
            else:
                reward = -10

        next_position = (nr, nc)

        future = 0 if done else np.max(
            Q[nr, nc, new_passenger]
        )

        Q[r, c, passenger, action] += alpha * (
            reward + gamma * future
            - Q[r, c, passenger, action]
        )

        position = next_position
        passenger = new_passenger

        if done:
            break

# Testing
position = start
passenger = 0
path = [position]

print("\nTaxi Starts At :", position)

for step in range(50):
    r, c = position

    action = np.argmax(Q[r, c, passenger])

    if action < 4:
        dr, dc = actions[action]
        nr, nc = r + dr, c + dc

        if not (0 <= nr < 4 and 0 <= nc < 4):
            break

        position = (nr, nc)
        path.append(position)

        print("Taxi Moves To :", position)

    elif action == 4 and position == pickup and passenger == 0:
        passenger = 1
        print("Passenger Picked Up!")

    elif action == 5 and position == drop and passenger == 1:
        print("Passenger Dropped Successfully!")
        break

    else:
        print("Invalid Action!")
        break

print("\nLearned Path :", path)

if position == drop and passenger == 1:
    print("Destination Reached Successfully !!")
else:
    print("Destination Not Reached !!")


Enter the row for taxi start(0-3) :  2
Enter the column for taxi start(0-3) :  1
Enter the row for passenger(0-3) :  0
Enter the column for passenger(0-3) :  0
Enter the row for destination(0-3) :  3
Enter the column for destination(0-3) :  2



Taxi Grid World :
P . . .
. . . .
. S . .
. . D .

Taxi Starts At : (2, 1)
Taxi Moves To : (1, 1)
Taxi Moves To : (0, 1)
Taxi Moves To : (0, 0)
Passenger Picked Up!
Taxi Moves To : (1, 0)
Taxi Moves To : (1, 1)
Taxi Moves To : (2, 1)
Taxi Moves To : (3, 1)
Taxi Moves To : (3, 2)
Passenger Dropped Successfully!

Learned Path : [(2, 1), (1, 1), (0, 1), (0, 0), (1, 0), (1, 1), (2, 1), (3, 1), (3, 2)]
Destination Reached Successfully !!
